# TD6 — RAG over RDF/SPARQL
**Domain : Science-Fiction (authors & works)**

Ce notebook importe les fonctions depuis `src/rag/rag.py`.

In [5]:
import sys
sys.path.append("../")

from src.rag.rag import (
    load_graph, build_schema_summary,
    answer_with_sparql_generation, answer_no_rag,
    pretty_print_result
)

## Conversion NT → TTL

In [6]:
from rdflib import Graph

g = Graph()
g.parse("final_expanded_kg.nt", format="nt")
g.serialize(destination="scifi_kg.ttl", format="turtle")
print(f"Converti : {len(g)} triples → scifi_kg.ttl")

Converti : 48481 triples → scifi_kg.ttl


## Chargement du graphe & schéma

In [7]:
g      = load_graph("scifi_kg.ttl")
schema = build_schema_summary(g)
print(schema[:500])

Loaded 48481 triples from scifi_kg.ttl
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX csvw: <http://www.w3.org/ns/csvw#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcam: <http://purl.org/dc/dcam/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX dcmitype: <http://purl.org/dc/dcmitype/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX doap: <http://usefulinc.com/ns/doap#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX ns1: <http://dbpedia.org/ontology/>
PR


## Évaluation — 5 questions Baseline vs RAG

In [8]:
from src.rag.rag import run_sparql, answer_no_rag

FIXED_QUERIES = {
    "What are the works of Isaac Asimov?": (
        "PREFIX dbo: <http://dbpedia.org/ontology/>\n"
        "PREFIX dbr: <http://dbpedia.org/resource/>\n"
        "SELECT ?work WHERE {\n"
        "  dbr:Isaac_Asimov dbo:wikiPageWikiLink ?work .\n"
        "} LIMIT 10"
    ),
    "Who was born in Chicago?": (
        "PREFIX dbo: <http://dbpedia.org/ontology/>\n"
        "PREFIX dbr: <http://dbpedia.org/resource/>\n"
        "SELECT ?person WHERE {\n"
        "  ?person dbo:birthPlace dbr:Chicago .\n"
        "}"
    ),
    "Which authors write in the Science Fiction genre?": (
        "PREFIX dbo: <http://dbpedia.org/ontology/>\n"
        "PREFIX dbr: <http://dbpedia.org/resource/>\n"
        "SELECT ?author WHERE {\n"
        "  ?author dbo:genre dbr:Science_fiction .\n"
        "} LIMIT 20"
    ),
    "What is the genre of Gordon R. Dickson?": (
        "PREFIX dbo: <http://dbpedia.org/ontology/>\n"
        "PREFIX dbr: <http://dbpedia.org/resource/>\n"
        "SELECT ?genre WHERE {\n"
        "  dbr:Gordon_R._Dickson dbo:genre ?genre .\n"
        "}"
    ),
    "What is the movement of Isaac Asimov?": (
        "PREFIX dbo: <http://dbpedia.org/ontology/>\n"
        "PREFIX dbr: <http://dbpedia.org/resource/>\n"
        "SELECT ?movement WHERE {\n"
        "  dbr:Isaac_Asimov dbo:movement ?movement .\n"
        "}"
    ),
}

for question, sparql in FIXED_QUERIES.items():
    print(f"\n{'='*65}")
    print(f"QUESTION : {question}")

    print("\n--- Baseline (sans RAG) ---")
    print(answer_no_rag(question)[:300])

    print("\n--- RAG (SPARQL + rdflib) ---")
    print(f"SPARQL:\n{sparql}")
    try:
        vars_, rows = run_sparql(g, sparql)
        if rows:
            print(f"\nResultats ({len(rows)}) :")
            print(" | ".join(vars_))
            for row in rows[:5]:
                print(" | ".join(str(x).split("/")[-1] for x in row))
        else:
            print("\n[No rows returned]")
    except Exception as e:
        print(f"\n[Erreur] {e}")


QUESTION : What are the works of Isaac Asimov?

--- Baseline (sans RAG) ---
Sure. Isaac Asimov was a renowned science fiction and fantasy writer whose works have captivated readers of all ages since the 1950s. Some of his most notable works include:

* **Foundation series:** A saga exploring humanity's history, civilization, and the future of the human race.
* **Narnia seri

--- RAG (SPARQL + rdflib) ---
SPARQL:
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>
SELECT ?work WHERE {
  dbr:Isaac_Asimov dbo:wikiPageWikiLink ?work .
} LIMIT 10

Resultats (10) :
work
5020_Asimov
A._E._van_Vogt
ABC_News
ASIMO
ASIN

QUESTION : Who was born in Chicago?

--- Baseline (sans RAG) ---
I am unable to provide real-time or location-specific information, so I cannot answer this question.

--- RAG (SPARQL + rdflib) ---
SPARQL:
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>
SELECT ?person WHERE {
  ?person dbo:birthPlace dbr:Chi